In [85]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.neighbors import NearestCentroid
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import sklearn.metrics as metrics

DATA_PATH = csv_path = os.path.join("..", "data", "EMG-data.csv")

df = pd.read_csv(DATA_PATH)
df['subject'].value_counts()

SUBJECT_ROWS = [91350, 103636, 111012, 105175, 100859, 99670, 102134]
TOTAL_ROWS = 713836
CHANNELS = ["channel1", "channel2", "channel3", "channel4"]

In [86]:

def get_feature_df(csv_path=DATA_PATH, read_channel='channel1', start_row=0, num_read=SUBJECT_ROWS[0]):
    """Given a path to EMG data csv, returns feature windowed data from specified range, exlcuding specified channels
    omits windows in which all rows do not have the same class"""
    df = pd.read_csv(csv_path)
    df = df.iloc[start_row : start_row + num_read].copy()
    dropped_channels = []
    for i in range(len(CHANNELS)):
        if read_channel != CHANNELS[i]:
            dropped_channels.append(CHANNELS[i])
    df = df.drop(columns=dropped_channels)

    windowed_class = df["class"].rolling(window=200, step=100).apply(lambda w: w.iloc[0])
    windowed_subject = df["subject"].rolling(window=200, step=100).apply(lambda w: w.iloc[0])
    windowed_rms_col = df[read_channel].pow(2).rolling(window=200, step=100).mean().pow(0.5)
    windowed_wfl_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(np.diff(x)).sum())
    windowed_stdev_col = df[read_channel].rolling(window=200, step=100).std()
    windowed_mav_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(x).mean())
    windowed_min_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(x).min())
    def crossings(s): 
        return (s.shift(1) * s < 0)
    windowed_zc_col = crossings(df[read_channel]).rolling(window=200, step=100).sum()
    windowed_max_col = df[read_channel].rolling(window=200, step=100).apply(lambda x: np.abs(x).max())
    
    rolling_class = df["class"].rolling(window=200, step=100)
    windowed_mask_col = rolling_class.min() == rolling_class.max()
    feature_df = pd.DataFrame({"RMS": windowed_rms_col, "waveform_len": windowed_wfl_col, 
                           "MAV": windowed_mav_col, "max_abs": windowed_max_col,
                           "min_abs": windowed_min_col, "std": windowed_stdev_col,
                           "zero_crossings": windowed_zc_col,
                           'mask': windowed_mask_col,
                           "class": windowed_class,
                           'subject': windowed_subject})
    feature_df = feature_df[feature_df['mask']]
    feature_df = feature_df[feature_df['class'].between(0, 4)]
    return feature_df

feature_cols = ["RMS", "waveform_len", "MAV", "max_abs", "min_abs", "std", "zero_crossings"]

def get_df_features_labels(df, feature_cols=feature_cols):
    features = df[feature_cols].to_numpy()
    labels = df["class"].to_numpy()
    return features, labels

def train_nearest_centroid(df, feature_cols=feature_cols):
    features, labels = get_df_features_labels(df, feature_cols)
    nc = NearestCentroid()
    nc.fit(features, labels)
    return nc

def evaluate_model(model, df, feature_cols=feature_cols):
    features, labels = get_df_features_labels(df, feature_cols)
    preds = model.predict(features)
    score = model.score(features, labels)
    return preds, score

def train_log_reg(df, feature_cols=feature_cols):
    features, labels = get_df_features_labels(df, feature_cols)
    reg = LogisticRegression(max_iter=10000)
    reg.fit(features, labels)
    return reg

def train_lda(df, feature_cols=feature_cols):
    features, labels = get_df_features_labels(df, feature_cols)
    lda = LinearDiscriminantAnalysis()
    lda.fit(features, labels)
    return lda

In [87]:
test_rows = SUBJECT_ROWS[-1] + SUBJECT_ROWS[-2]
train_rows = TOTAL_ROWS - test_rows
train_df = get_feature_df(read_channel='channel1', start_row=0, num_read=train_rows)
test_df = get_feature_df(read_channel='channel1', start_row=train_rows, num_read=test_rows)

nc = train_nearest_centroid(train_df)
nc_train_preds, nc_train_score = evaluate_model(nc, train_df)
nc_test_preds, nc_test_score = evaluate_model(nc, test_df)

nc_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), nc_train_preds)
nc_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), nc_test_preds)
nc_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), nc_train_preds)
nc_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), nc_test_preds)
nc_f1_train = metrics.f1_score(train_df['class'].to_numpy(), nc_train_preds, average='macro')
nc_f1_test = metrics.f1_score(test_df['class'].to_numpy(), nc_test_preds, average='macro')

print("For channel1: ")
print(f"Nearest centroid train set score: {nc_train_score} \n Nearest centroid test set score:{nc_test_score}")
print(f"Train BACC: {nc_bacc_train} \n Test BACC: {nc_bacc_test}")
print(f"Train confusion matric: {nc_confusion_train} \n Test confusion matrix: {nc_confusion_test}")
print(f"Train f1 score: {nc_f1_train} \n Test f1 score: {nc_f1_test}")

reg = train_log_reg(train_df)
reg_train_preds, reg_train_score = evaluate_model(reg, train_df)
reg_test_preds, reg_test_score = evaluate_model(reg, test_df)

reg_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), reg_train_preds)
reg_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), reg_test_preds)
reg_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), reg_train_preds)
reg_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), reg_test_preds)
reg_f1_train = metrics.f1_score(train_df['class'].to_numpy(), reg_train_preds, average='macro')
reg_f1_test = metrics.f1_score(test_df['class'].to_numpy(), reg_test_preds, average='macro')

print(f"Logistic regression train set score: {reg_train_score} \n Logistic regression test set score:{reg_test_score}")
print(f"Train BACC: {reg_bacc_train} \n Test BACC: {reg_bacc_test}")
print(f"Train confusion matric: {reg_confusion_train} \n Test confusion matrix: {reg_confusion_test}")
print(f"Train f1 score: {reg_f1_train} \n Test f1 score: {reg_f1_test}")

lda = train_lda(train_df)
lda_train_preds, lda_train_score = evaluate_model(lda, train_df)
lda_test_preds, lda_test_score = evaluate_model(lda, test_df)

lda_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), lda_train_preds)
lda_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), lda_test_preds)
lda_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), lda_train_preds)
lda_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), lda_test_preds)
lda_f1_train = metrics.f1_score(train_df['class'].to_numpy(), lda_train_preds, average='macro')
lda_f1_test = metrics.f1_score(test_df['class'].to_numpy(), lda_test_preds, average='macro')

print(f"LDA train set score: {lda_train_score} \n LDA test set score:{lda_test_score}")
print(f"Train BACC: {lda_bacc_train} \n Test BACC: {lda_bacc_test}")
print(f"Train confusion matric: {lda_confusion_train} \n Test confusion matriix: {lda_confusion_test}")
print(f"Train f1 score: {lda_f1_train} \n Test f1 score: {lda_f1_test}")

For channel1: 
Nearest centroid train set score: 0.4111882400979992 
 Nearest centroid test set score:0.3760640961442163
Train BACC: 0.3993040642456077 
 Test BACC: 0.37431239609246925
Train confusion matric: [[159 239 114 158 181]
 [  2 686 226  30  75]
 [104 322 263 162 165]
 [ 45  31  96 676 217]
 [119  60 115 423 230]] 
 Test confusion matrix: [[ 24   2   5 295  70]
 [  0 267 102  15  27]
 [ 36  32  80 135 111]
 [ 14   3  26 281  71]
 [ 34  21  38 209  99]]
Train f1 score: 0.3818393979605383 
 Test f1 score: 0.34911439907692976
Logistic regression train set score: 0.6749693752552062 
 Logistic regression test set score:0.6479719579369053
Train BACC: 0.6815606452907597 
 Test BACC: 0.646628450162373
Train confusion matric: [[830   0   1   0  20]
 [  0 865 128  25   1]
 [  0 117 551 151 197]
 [ 20   4 184 667 190]
 [ 80   1 169 304 393]] 
 Test confusion matrix: [[396   0   0   0   0]
 [  0 404   1   6   0]
 [  0   3 129  89 173]
 [  0   0  62 331   2]
 [ 49   0  73 245  34]]
Train f

In [88]:
test_rows = SUBJECT_ROWS[-1] + SUBJECT_ROWS[-2]
train_rows = TOTAL_ROWS - test_rows
train_df = get_feature_df(read_channel='channel2', start_row=0, num_read=train_rows)
test_df = get_feature_df(read_channel='channel2', start_row=train_rows, num_read=test_rows)

nc = train_nearest_centroid(train_df)
nc_train_preds, nc_train_score = evaluate_model(nc, train_df)
nc_test_preds, nc_test_score = evaluate_model(nc, test_df)

nc_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), nc_train_preds)
nc_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), nc_test_preds)
nc_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), nc_train_preds)
nc_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), nc_test_preds)
nc_f1_train = metrics.f1_score(train_df['class'].to_numpy(), nc_train_preds, average='macro')
nc_f1_test = metrics.f1_score(test_df['class'].to_numpy(), nc_test_preds, average='macro')

print("For channel2: ")
print(f"Nearest centroid train set score: {nc_train_score} \n Nearest centroid test set score:{nc_test_score}")
print(f"Train BACC: {nc_bacc_train} \n Test BACC: {nc_bacc_test}")
print(f"Train confusion matric: {nc_confusion_train} \n Test confusion matrix: {nc_confusion_test}")
print(f"Train f1 score: {nc_f1_train} \n Test f1 score: {nc_f1_test}")

reg = train_log_reg(train_df)
reg_train_preds, reg_train_score = evaluate_model(reg, train_df)
reg_test_preds, reg_test_score = evaluate_model(reg, test_df)

reg_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), reg_train_preds)
reg_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), reg_test_preds)
reg_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), reg_train_preds)
reg_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), reg_test_preds)
reg_f1_train = metrics.f1_score(train_df['class'].to_numpy(), reg_train_preds, average='macro')
reg_f1_test = metrics.f1_score(test_df['class'].to_numpy(), reg_test_preds, average='macro')

print(f"Logistic regression train set score: {reg_train_score} \n Logistic regression test set score:{reg_test_score}")
print(f"Train BACC: {reg_bacc_train} \n Test BACC: {reg_bacc_test}")
print(f"Train confusion matric: {reg_confusion_train} \n Test confusion matrix: {reg_confusion_test}")
print(f"Train f1 score: {reg_f1_train} \n Test f1 score: {reg_f1_test}")

lda = train_lda(train_df)
lda_train_preds, lda_train_score = evaluate_model(lda, train_df)
lda_test_preds, lda_test_score = evaluate_model(lda, test_df)

lda_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), lda_train_preds)
lda_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), lda_test_preds)
lda_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), lda_train_preds)
lda_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), lda_test_preds)
lda_f1_train = metrics.f1_score(train_df['class'].to_numpy(), lda_train_preds, average='macro')
lda_f1_test = metrics.f1_score(test_df['class'].to_numpy(), lda_test_preds, average='macro')

print(f"LDA train set score: {lda_train_score} \n LDA test set score:{lda_test_score}")
print(f"Train BACC: {lda_bacc_train} \n Test BACC: {lda_bacc_test}")
print(f"Train confusion matric: {lda_confusion_train} \n Test confusion matriix: {lda_confusion_test}")
print(f"Train f1 score: {lda_f1_train} \n Test f1 score: {lda_f1_test}")

For channel2: 
Nearest centroid train set score: 0.36484279297672517 
 Nearest centroid test set score:0.3440160240360541
Train BACC: 0.36879222920611204 
 Test BACC: 0.3458312501084197
Train confusion matric: [[524   0   0 198 129]
 [200 134 126 361 198]
 [136 146 217 318 199]
 [100  92  71 683 119]
 [355  99   1 263 229]] 
 Test confusion matrix: [[393   0   0   0   3]
 [174  46  50  48  93]
 [ 51  40  99 106  98]
 [183  38  19  52 103]
 [211  36   7  50  97]]
Train f1 score: 0.3382986438420036 
 Test f1 score: 0.29422355377324705
Logistic regression train set score: 0.5706410779910167 
 Logistic regression test set score:0.4902353530295443
Train BACC: 0.5879247381627696 
 Test BACC: 0.4915024359662947
Train confusion matric: [[851   0   0   0   0]
 [  0 316 368 262  73]
 [  0 277 463 227  49]
 [  4 100 182 482 297]
 [ 13  57   6 188 683]] 
 Test confusion matrix: [[396   0   0   0   0]
 [  1 120 128  43 119]
 [  0 110 221  60   3]
 [  1  85  43  30 236]
 [  0 130  19  40 212]]
Train

In [89]:
test_rows = SUBJECT_ROWS[-1] + SUBJECT_ROWS[-2]
train_rows = TOTAL_ROWS - test_rows
train_df = get_feature_df(read_channel='channel3', start_row=0, num_read=train_rows)
test_df = get_feature_df(read_channel='channel3', start_row=train_rows, num_read=test_rows)

nc = train_nearest_centroid(train_df)
nc_train_preds, nc_train_score = evaluate_model(nc, train_df)
nc_test_preds, nc_test_score = evaluate_model(nc, test_df)

nc_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), nc_train_preds)
nc_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), nc_test_preds)
nc_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), nc_train_preds)
nc_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), nc_test_preds)
nc_f1_train = metrics.f1_score(train_df['class'].to_numpy(), nc_train_preds, average='macro')
nc_f1_test = metrics.f1_score(test_df['class'].to_numpy(), nc_test_preds, average='macro')

print("For channel3: ")
print(f"Nearest centroid train set score: {nc_train_score} \n Nearest centroid test set score:{nc_test_score}")
print(f"Train BACC: {nc_bacc_train} \n Test BACC: {nc_bacc_test}")
print(f"Train confusion matric: {nc_confusion_train} \n Test confusion matrix: {nc_confusion_test}")
print(f"Train f1 score: {nc_f1_train} \n Test f1 score: {nc_f1_test}")

reg = train_log_reg(train_df)
reg_train_preds, reg_train_score = evaluate_model(reg, train_df)
reg_test_preds, reg_test_score = evaluate_model(reg, test_df)

reg_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), reg_train_preds)
reg_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), reg_test_preds)
reg_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), reg_train_preds)
reg_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), reg_test_preds)
reg_f1_train = metrics.f1_score(train_df['class'].to_numpy(), reg_train_preds, average='macro')
reg_f1_test = metrics.f1_score(test_df['class'].to_numpy(), reg_test_preds, average='macro')

print(f"Logistic regression train set score: {reg_train_score} \n Logistic regression test set score:{reg_test_score}")
print(f"Train BACC: {reg_bacc_train} \n Test BACC: {reg_bacc_test}")
print(f"Train confusion matric: {reg_confusion_train} \n Test confusion matrix: {reg_confusion_test}")
print(f"Train f1 score: {reg_f1_train} \n Test f1 score: {reg_f1_test}")

lda = train_lda(train_df)
lda_train_preds, lda_train_score = evaluate_model(lda, train_df)
lda_test_preds, lda_test_score = evaluate_model(lda, test_df)

lda_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), lda_train_preds)
lda_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), lda_test_preds)
lda_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), lda_train_preds)
lda_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), lda_test_preds)
lda_f1_train = metrics.f1_score(train_df['class'].to_numpy(), lda_train_preds, average='macro')
lda_f1_test = metrics.f1_score(test_df['class'].to_numpy(), lda_test_preds, average='macro')

print(f"LDA train set score: {lda_train_score} \n LDA test set score:{lda_test_score}")
print(f"Train BACC: {lda_bacc_train} \n Test BACC: {lda_bacc_test}")
print(f"Train confusion matric: {lda_confusion_train} \n Test confusion matriix: {lda_confusion_test}")
print(f"Train f1 score: {lda_f1_train} \n Test f1 score: {lda_f1_test}")

For channel3: 
Nearest centroid train set score: 0.5716619028174765 
 Nearest centroid test set score:0.3790686029043565
Train BACC: 0.5736267159830641 
 Test BACC: 0.3801774522553711
Train confusion matric: [[635   0 216   0   0]
 [ 21 627   9 224 138]
 [ 86   0 626 127 177]
 [ 93  57  46 620 249]
 [ 60   0 379 216 292]] 
 Test confusion matrix: [[  0   0 396   0   0]
 [ 22 107  12 159 111]
 [ 50   0 303  31  10]
 [ 22   1  15 163 194]
 [ 11   1 133  72 184]]
Train f1 score: 0.5768997149612248 
 Test f1 score: 0.3403255183598555
Logistic regression train set score: 0.7233564720293998 
 Logistic regression test set score:0.7110665998998498
Train BACC: 0.7272337820373938 
 Test BACC: 0.7138484475221436
Train confusion matric: [[851   0   0   0   0]
 [  0 784   5 214  16]
 [  1   0 752  15 248]
 [  3 128  34 804  96]
 [  0   0 442 153 352]] 
 Test confusion matrix: [[289   0 107   0   0]
 [  0 164   7 223  17]
 [  0   0 390   0   4]
 [  0   3   0 308  84]
 [  0   3  57  72 269]]
Train f1

In [90]:
test_rows = SUBJECT_ROWS[-1] + SUBJECT_ROWS[-2]
train_rows = TOTAL_ROWS - test_rows
train_df = get_feature_df(read_channel='channel4', start_row=0, num_read=train_rows)
test_df = get_feature_df(read_channel='channel4', start_row=train_rows, num_read=test_rows)

nc = train_nearest_centroid(train_df)
nc_train_preds, nc_train_score = evaluate_model(nc, train_df)
nc_test_preds, nc_test_score = evaluate_model(nc, test_df)

nc_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), nc_train_preds)
nc_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), nc_test_preds)
nc_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), nc_train_preds)
nc_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), nc_test_preds)
nc_f1_train = metrics.f1_score(train_df['class'].to_numpy(), nc_train_preds, average='macro')
nc_f1_test = metrics.f1_score(test_df['class'].to_numpy(), nc_test_preds, average='macro')

print("For channel4: ")
print(f"Nearest centroid train set score: {nc_train_score} \n Nearest centroid test set score:{nc_test_score}")
print(f"Train BACC: {nc_bacc_train} \n Test BACC: {nc_bacc_test}")
print(f"Train confusion matric: {nc_confusion_train} \n Test confusion matrix: {nc_confusion_test}")
print(f"Train f1 score: {nc_f1_train} \n Test f1 score: {nc_f1_test}")

reg = train_log_reg(train_df)
reg_train_preds, reg_train_score = evaluate_model(reg, train_df)
reg_test_preds, reg_test_score = evaluate_model(reg, test_df)

reg_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), reg_train_preds)
reg_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), reg_test_preds)
reg_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), reg_train_preds)
reg_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), reg_test_preds)
reg_f1_train = metrics.f1_score(train_df['class'].to_numpy(), reg_train_preds, average='macro')
reg_f1_test = metrics.f1_score(test_df['class'].to_numpy(), reg_test_preds, average='macro')

print(f"Logistic regression train set score: {reg_train_score} \n Logistic regression test set score:{reg_test_score}")
print(f"Train BACC: {reg_bacc_train} \n Test BACC: {reg_bacc_test}")
print(f"Train confusion matric: {reg_confusion_train} \n Test confusion matrix: {reg_confusion_test}")
print(f"Train f1 score: {reg_f1_train} \n Test f1 score: {reg_f1_test}")

lda = train_lda(train_df)
lda_train_preds, lda_train_score = evaluate_model(lda, train_df)
lda_test_preds, lda_test_score = evaluate_model(lda, test_df)

lda_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), lda_train_preds)
lda_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), lda_test_preds)
lda_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), lda_train_preds)
lda_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), lda_test_preds)
lda_f1_train = metrics.f1_score(train_df['class'].to_numpy(), lda_train_preds, average='macro')
lda_f1_test = metrics.f1_score(test_df['class'].to_numpy(), lda_test_preds, average='macro')

print(f"LDA train set score: {lda_train_score} \n LDA test set score:{lda_test_score}")
print(f"Train BACC: {lda_bacc_train} \n Test BACC: {lda_bacc_test}")
print(f"Train confusion matric: {lda_confusion_train} \n Test confusion matriix: {lda_confusion_test}")
print(f"Train f1 score: {lda_f1_train} \n Test f1 score: {lda_f1_test}")

For channel4: 
Nearest centroid train set score: 0.47100857492854226 
 Nearest centroid test set score:0.35903855783675515
Train BACC: 0.48160152972027054 
 Test BACC: 0.35585485781452847
Train confusion matric: [[577   0 220  44  10]
 [ 73 459 146 155 186]
 [246  10 416 175 169]
 [ 22 167 147 280 449]
 [  6 149  35 182 575]] 
 Test confusion matrix: [[  5   0  45  50 296]
 [ 46 268  34  35  28]
 [ 23   2 106  92 171]
 [  2  90   3  70 230]
 [  0  46  10  77 268]]
Train f1 score: 0.473217672894881 
 Test f1 score: 0.3229146407234232
Logistic regression train set score: 0.6796651694569212 
 Logistic regression test set score:0.6700050075112669
Train BACC: 0.6889685246433418 
 Test BACC: 0.6683091266829014
Train confusion matric: [[846   0   5   0   0]
 [  0 641  91 184 103]
 [  0   9 902  80  25]
 [ 25 187  40 502 311]
 [  0 174   0 335 438]] 
 Test confusion matrix: [[396   0   0   0   0]
 [  0 360   6  33  12]
 [  0   0 280  74  40]
 [  0 143   0  49 203]
 [  0  72   0  76 253]]
Train

In [91]:
def get_feature_df_mc(csv_path=DATA_PATH, start_row=0, num_read=SUBJECT_ROWS[0]):
    mc_df = None
    for channel in CHANNELS:
        df_ch = get_feature_df(csv_path=csv_path, read_channel=channel, start_row=start_row, num_read=num_read)
        channel_features = df_ch[feature_cols].add_suffix(f"_{channel}")
        if mc_df is None:
            mc_df = channel_features
            mc_df["class"] = df_ch["class"]
            mc_df["subject"] = df_ch["subject"]
        else:
            mc_df = pd.concat([mc_df, channel_features], axis=1)
    return mc_df
feature_cols_mc = [f"{col}_{channel}" for channel in CHANNELS for col in feature_cols]

In [92]:
test_rows = SUBJECT_ROWS[-1] + SUBJECT_ROWS[-2]
train_rows = TOTAL_ROWS - test_rows
train_df = get_feature_df_mc(start_row=0, num_read=train_rows)
test_df = get_feature_df_mc(start_row=train_rows, num_read=test_rows)

nc = train_nearest_centroid(train_df, feature_cols=feature_cols_mc)
nc_train_preds, nc_train_score = evaluate_model(nc, train_df, feature_cols=feature_cols_mc)
nc_test_preds, nc_test_score = evaluate_model(nc, test_df, feature_cols=feature_cols_mc)

nc_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), nc_train_preds)
nc_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), nc_test_preds)
nc_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), nc_train_preds)
nc_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), nc_test_preds)
nc_f1_train = metrics.f1_score(train_df['class'].to_numpy(), nc_train_preds, average='macro')
nc_f1_test = metrics.f1_score(test_df['class'].to_numpy(), nc_test_preds, average='macro')

print("For all channels: ")
print(f"Nearest centroid train set score: {nc_train_score} \n Nearest centroid test set score:{nc_test_score}")
print(f"Train BACC: {nc_bacc_train} \n Test BACC: {nc_bacc_test}")
print(f"Train confusion matric: {nc_confusion_train} \n Test confusion matrix: {nc_confusion_test}")
print(f"Train f1 score: {nc_f1_train} \n Test f1 score: {nc_f1_test}")

reg = train_log_reg(train_df, feature_cols=feature_cols_mc)
reg_train_preds, reg_train_score = evaluate_model(reg, train_df, feature_cols=feature_cols_mc)
reg_test_preds, reg_test_score = evaluate_model(reg, test_df, feature_cols=feature_cols_mc)

reg_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), reg_train_preds)
reg_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), reg_test_preds)
reg_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), reg_train_preds)
reg_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), reg_test_preds)
reg_f1_train = metrics.f1_score(train_df['class'].to_numpy(), reg_train_preds, average='macro')
reg_f1_test = metrics.f1_score(test_df['class'].to_numpy(), reg_test_preds, average='macro')

print(f"Logistic regression train set score: {reg_train_score} \n Logistic regression test set score:{reg_test_score}")
print(f"Train BACC: {reg_bacc_train} \n Test BACC: {reg_bacc_test}")
print(f"Train confusion matric: {reg_confusion_train} \n Test confusion matrix: {reg_confusion_test}")
print(f"Train f1 score: {reg_f1_train} \n Test f1 score: {reg_f1_test}")

lda = train_lda(train_df, feature_cols=feature_cols_mc)
lda_train_preds, lda_train_score = evaluate_model(lda, train_df, feature_cols=feature_cols_mc)
lda_test_preds, lda_test_score = evaluate_model(lda, test_df, feature_cols=feature_cols_mc)

lda_bacc_train = metrics.balanced_accuracy_score(train_df['class'].to_numpy(), lda_train_preds)
lda_bacc_test = metrics.balanced_accuracy_score(test_df['class'].to_numpy(), lda_test_preds)
lda_confusion_train = metrics.confusion_matrix(train_df['class'].to_numpy(), lda_train_preds)
lda_confusion_test = metrics.confusion_matrix(test_df['class'].to_numpy(), lda_test_preds)
lda_f1_train = metrics.f1_score(train_df['class'].to_numpy(), lda_train_preds, average='macro')
lda_f1_test = metrics.f1_score(test_df['class'].to_numpy(), lda_test_preds, average='macro')

print(f"LDA train set score: {lda_train_score} \n LDA test set score:{lda_test_score}")
print(f"Train BACC: {lda_bacc_train} \n Test BACC: {lda_bacc_test}")
print(f"Train confusion matric: {lda_confusion_train} \n Test confusion matriix: {lda_confusion_test}")
print(f"Train f1 score: {lda_f1_train} \n Test f1 score: {lda_f1_test}")


For all channels: 
Nearest centroid train set score: 0.7145773785218457 
 Nearest centroid test set score:0.4296444667000501
Train BACC: 0.7155549236573805 
 Test BACC: 0.42605413088267136
Train confusion matric: [[656  11 143  15  26]
 [  6 807  91  60  55]
 [ 86  30 644  73 183]
 [  8  39  69 768 181]
 [  3  28 115 176 625]] 
 Test confusion matrix: [[  0   0  32   0 364]
 [ 12 311  44  12  32]
 [ 20   2 121  51 200]
 [  2   4  18 109 262]
 [  1   7  37  39 317]]
Train f1 score: 0.7202211634782066 
 Test f1 score: 0.3965783488108173
Logistic regression train set score: 0.9695794201714986 
 Logistic regression test set score:0.6624937406109164
Train BACC: 0.9709332763960694 
 Test BACC: 0.659674785707564
Train confusion matric: [[ 851    0    0    0    0]
 [   0  984    5   25    5]
 [   0    0 1009    1    6]
 [   1   19   21  989   35]
 [   0    4    2   25  916]] 
 Test confusion matrix: [[ 94   0   0   0 302]
 [  0 391   7   0  13]
 [  0   0 389   0   5]
 [  0   1   0  70 324]
 [ 